# LLM-as-a-judge

In [1]:
import os
import json
import re
from groq import Groq

In [13]:
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [29]:
# Use a supported model name string, not a SentenceTransformer object
def evaluate_with_llm_judge(query, context, generated_answer, model="mixtral-8x7b-32768"):
    if not client:
        return {
            "error": "LLM client not initialized. Please set up the client before calling this function."
        }
    # Ensure model is a string, not a SentenceTransformer object
    if not isinstance(model, str):
        raise ValueError("The 'model' argument must be a string with the model name, not a SentenceTransformer object.")
    prompt = f"""You are an impartial judge evaluating the quality of an answer generated by a RAG system.
    Your task is to evaluate the generated answer based on the following criteria:
    1. **Faithfulness**: Does the answer accurately reflect the information provided in the retrieved documents? Are there any hallucinations or inaccuracies?
    2. **Relevance**: Does the answer directly address the user's query? Is the information provided pertinent to the question asked?
    You must provide a score from 1 to 5 for each criterion, where 1 is poor and 5 is excellent. Additionally, provide a brief explanation for each score.
    Here is the user's query:
    {query}
    Here is the context retrieved from the documents:
    {context}
    Here is the answer generated by the RAG system:
    {generated_answer}
    Please provide your evaluation only in a valid JSON format with the following keys: "faithfulness_score", "faithfulness_explanation", "relevance_score", "relevance_explanation".
    """

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an impartial judge evaluating the quality of an answer generated by a RAG system."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
        )
        response_text = response.choices[0].message.content
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            evaluation = json.loads(json_str)
            return evaluation
        else:
            print("No valid JSON found in the response.")
            return None
    except Exception as e:
        print(f"Error during LLM evaluation: {e}")
        return None


In [30]:
evaluation_data = {
    "query": "What are the main benefits of using a RAG system for question answering?",
    "context": "RAG systems combine retrieval and generation to provide accurate and relevant answers. They retrieve relevant documents from a knowledge base and then generate answers based on that information. This approach allows for more accurate and contextually relevant responses compared to traditional language models that rely solely on pre-trained knowledge.",
    "generated_answer": "The main benefits of using a RAG system for question answering include improved accuracy, as it retrieves relevant information from a knowledge base, and enhanced relevance, as it generates answers based on the retrieved context. This leads to more informative and contextually appropriate responses compared to traditional language models."
}
evaluation_result = evaluate_with_llm_judge(
    query=evaluation_data["query"],
    context=evaluation_data["context"],
    generated_answer=evaluation_data["generated_answer"]
)

if evaluation_result is not None:
    print(f"Faithfulness Score: {evaluation_result.get('faithfulness_score', 'N/A')}")
    print(f"Relevance Score: {evaluation_result.get('relevance_score', 'N/A')}")
    print(f"Faithfulness Explanation: {evaluation_result.get('faithfulness_explanation', 'N/A')}")
    print(f"Relevance Explanation: {evaluation_result.get('relevance_explanation', 'N/A')}")
else:
    print("Evaluation failed or no valid result returned.")


Error during LLM evaluation: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Evaluation failed or no valid result returned.


# UMBRELA metric


In [ ]:
UMBRELA_prompt = """
Given a query and a passage, you must provide a score on an integer scale from 0 to 3 
with the following criteria:
0: The passage does not contain any relevant information to answer the query.
1: The passage contains some relevant information, but it is incomplete or not directly related to the
query.
2: The passage contains relevant information that can help answer the query, but it may not be sufficient on its own.
3: The passage contains comprehensive and directly relevant information that can fully answer the query.

Important instructions:
Assign a category 1 is the passae is somewhat related to the query, but not directly relevant. For example, if the passage mentions a related topic but does not provide specific information that can be used to answer the query, it should be categorized as 1.
Assign a category 2 if the passage contains relevant information that can help answer the query, but it may not be sufficient on its own. For example, if the passage provides some details that are relevant to the query but does not cover all aspects needed for a complete answer, it should be categorized as 2.
Assign a category 3 if the passage contains comprehensive and directly relevant information that can fully answer the
query. For example, if the passage provides detailed information that directly addresses the query and can be used to construct a complete answer, it should be categorized as 3.
if none of the above criteria are met, assign a category 0. For example, if the passage does not mention any information that is relevant to the query or is completely unrelated, it should be categorized as 0.
Here is the query:
{query}
Here is the passage:
{chunk}

Split the problem into the following steps:
Consider the intent of the search
Measure how well the content matches the query
Measure how trustworthy the content is
Consider the aspects above and decide a final score.
Do not provide code in result.
"""